# R1-01 · Traer el dato: archivos, JSON y APIs — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea A · *Analista de Datos* · Semana 2 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Escribir **funciones** que cargan datos.
2. Leer **JSON** de una respuesta tipo API.
3. Aplicar el patrón **en vivo o caché** ante fallos de red.
4. **Guardar** resultados en archivos.

**Competencia de salida:** traer datos públicos desde archivos y APIs de forma robusta.

**Dato real:** compras públicas (ChileCompra) en CSV + respuestas JSON tipo API.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd
import json, urllib.request, os
import matplotlib.pyplot as plt

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Datos base cargados arriba. Aprendamos a traerlos de varias fuentes.

## 1. Funciones para cargar datos

Una **función** encapsula la carga para reutilizarla. Empezamos por leer un CSV con pandas.

### ✍️ Ejercicio 1 — Escribe `cargar_csv`

In [ ]:
def cargar_csv(ruta):
    return pd.read_csv(ruta)

datos = cargar_csv("compras_ml.csv")
print("Cargado:", datos.shape)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert cargar_csv("compras_ml.csv").shape == df.shape
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'pd.read_csv(ruta) y return.')
    print("   Detalle:", e)

## 2. JSON: el formato de las APIs

Las APIs responden **JSON**. Con `json.loads` se convierte en diccionarios y listas que recorres.

### ✍️ Ejercicio 2 — Parsea una respuesta y suma los montos

In [ ]:
respuesta = '{"results": [{"categoria": "pan", "monto": 500}, {"categoria": "leche", "monto": 800}]}'
data = json.loads(respuesta)
montos = [r["monto"] for r in data["results"]]
total = sum(montos)
print("registros:", len(data["results"]), "| total:", total)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert len(data["results"]) == 2
    assert total == 1300
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "json.loads(respuesta); luego [r['monto'] for r in data['results']] y sum(...).")
    print("   Detalle:", e)

## 3. En vivo o caché: APIs que no fallan

Una fuente remota puede caerse. El patrón **en vivo o caché**: intenta la fuente remota y, si falla, usa la copia local. Tu análisis siempre corre.

### ✍️ Ejercicio 3 — Escribe `cargar_o_cache`

In [ ]:
def cargar_o_cache(url, ruta_local):
    try:
        return pd.read_csv(url)
    except Exception:
        return pd.read_csv(ruta_local)

d = cargar_o_cache("http://no-existe.invalid/x.csv", "compras_ml.csv")
print("filas cargadas:", len(d))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(d) == len(df)
    assert callable(cargar_o_cache)
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'En el except: return pd.read_csv(ruta_local). La URL inválida fuerza el camino de caché.')
    print("   Detalle:", e)

## 4. Guardar el resultado

Tras traer y resumir, **persistes** el resultado en un archivo para compartirlo o reutilizarlo.

### ✍️ Ejercicio 4 — Guarda un resumen por categoría

In [ ]:
resumen = df.groupby("categoria")["monto_total"].sum().reset_index()
resumen.to_csv("resumen_categorias.csv", index=False)
existe = os.path.exists("resumen_categorias.csv")
print("guardado:", existe, "| filas:", len(resumen))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert existe is True
    assert pd.read_csv("resumen_categorias.csv").shape[0] == df["categoria"].nunique()
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "resumen.to_csv('resumen_categorias.csv', index=False).")
    print("   Detalle:", e)

In [ ]:
# (ilustración) El resumen que acabamos de guardar
r = resumen.sort_values("monto_total").tail(8)
plt.figure(figsize=(6, 4))
plt.barh(r["categoria"], r["monto_total"], color="#4240e5")
plt.xlabel("Gasto total"); plt.title("Gasto por categoría (top 8)")
plt.tight_layout(); plt.show()

## Cierre

- Encapsula la carga en **funciones** reutilizables.
- Las APIs hablan **JSON**: `json.loads` lo vuelve diccionarios y listas.
- El patrón **en vivo o caché** hace tu trabajo robusto ante caídas de red.
- **Guarda** tus resultados para compartirlos y reutilizarlos.

> *Con el dato ya en casa, en R1-02 empezamos a explorarlo con pandas.*